# 📖 Notebook 4: Security Gates in CI/CD

You've learned to find vulnerabilities manually. But what about the hundreds of developers on your team? You can't review every line of code by hand.

The answer is **automated security gates** — tools that run on every pull request and block insecure code from being merged.

At Microsoft, the SDL requires that **no code ships without passing automated security checks**. This notebook shows you what those checks look like.

## Learning Objectives

By the end of this notebook, you'll understand:
- What SAST (Static Analysis) is and how to run it
- What DAST (Dynamic Analysis) is and how it differs from SAST
- How to check dependencies for known vulnerabilities
- How to set up a security gate pipeline
- The security sign-off process at large enterprises

## 🛠️ Setup

```bash
cd enterprise-patterns/security-review
docker-compose up -d
source .venv/bin/activate
uv sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

## The Security Gate Pipeline

In a modern CI/CD pipeline, security checks run automatically:

```
Developer pushes code
        │
        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   Gate 1: SAST  │────▶│  Gate 2: Deps   │────▶│  Gate 3: DAST   │
│                 │     │                 │     │                 │
│ Scan source     │     │ Check libraries │     │ Scan running    │
│ code for bugs   │     │ for known CVEs  │     │ app for vulns   │
│                 │     │                 │     │                 │
│ Tool: Bandit    │     │ Tool: safety    │     │ Tool: OWASP ZAP │
│ (Python)        │     │ pip-audit       │     │ Burp Suite      │
└────────┬────────┘     └────────┬────────┘     └────────┬────────┘
         │                       │                       │
    Pass/Fail              Pass/Fail              Pass/Fail
         │                       │                       │
         └───────────┬───────────┘───────────────────────┘
                     ▼
            ┌────────────────┐
            │ All gates pass │──▶ ✅ Merge allowed
            │ Any gate fails │──▶ ❌ Merge blocked
            └────────────────┘
                     │
                     ▼
            ┌────────────────┐
            │ Gate 4: Human  │
            │ Security       │
            │ Review         │
            │ (for critical  │
            │  changes)      │
            └────────────────┘
```

## Gate 1: SAST — Static Application Security Testing

**SAST** scans your **source code** without running it. It looks for patterns that are known to be dangerous.

| Feature | Details |
|---------|--------|
| **When it runs** | On every commit / pull request |
| **What it scans** | Source code, configuration files |
| **What it finds** | SQL injection, hardcoded secrets, insecure functions |
| **Speed** | Fast (seconds to minutes) |
| **False positives** | Moderate — some findings may not be exploitable |

### Popular SAST Tools

| Tool | Language | Free? |
|------|----------|-------|
| **Bandit** | Python | ✅ Open source |
| **Semgrep** | Many languages | ✅ Community rules |
| **CodeQL** | Many languages | ✅ Free for open source (GitHub) |
| **SonarQube** | Many languages | ✅ Community edition |
| **Checkmarx** | Many languages | ❌ Commercial |

In [ ]:
# === Gate 1: Run Bandit (Python SAST tool) on our vulnerable app ===
# Bandit is the most popular open-source SAST tool for Python.
# It checks for common security issues like:
# - SQL injection (B608)
# - Hardcoded passwords (B105, B106)
# - Use of exec/eval (B102)
# - Weak cryptography (B303)
# - Shell injection (B602)

import subprocess
import sys

print("🔍 Gate 1: SAST — Running Bandit on vulnerable_app.py")
print("=" * 70)

# Run Bandit and capture output
result = subprocess.run(
    [sys.executable, "-m", "bandit", "-r", "../app/vulnerable_app.py", "-f", "json"],
    capture_output=True,
    text=True,
)

import json
try:
    report = json.loads(result.stdout)
    results = report.get("results", [])
    
    print(f"\nFound {len(results)} security issues:\n")
    
    for issue in results:
        severity = issue["issue_severity"]
        confidence = issue["issue_confidence"]
        icon = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}.get(severity, "⚪")
        
        print(f"  {icon} [{severity}/{confidence}] {issue['test_id']}: {issue['issue_text']}")
        print(f"     Line {issue['line_number']}: {issue['code'].strip().split(chr(10))[0]}")
        print()
    
    # Summary
    metrics = report.get("metrics", {}).get("_totals", {})
    high = metrics.get("SEVERITY.HIGH", 0)
    medium = metrics.get("SEVERITY.MEDIUM", 0)
    low = metrics.get("SEVERITY.LOW", 0)
    
    print(f"Summary: {high} High, {medium} Medium, {low} Low")
    
    if high > 0:
        print("\n❌ GATE FAILED: High severity issues found. Merge blocked.")
    else:
        print("\n✅ GATE PASSED: No high severity issues.")
        
except json.JSONDecodeError:
    print("Bandit output:")
    print(result.stdout or result.stderr)

### Understanding Bandit Results

Each Bandit finding has:
- **Test ID** (e.g., B608): Identifies the specific check
- **Severity**: How dangerous the issue is (HIGH/MEDIUM/LOW)
- **Confidence**: How sure Bandit is that this is a real issue (HIGH/MEDIUM/LOW)

Common Bandit checks relevant to our app:

| ID | Check | What It Finds |
|----|-------|---------------|
| B105 | hardcoded_password_string | Passwords assigned to variables |
| B106 | hardcoded_password_funcarg | Passwords passed to function arguments |
| B303 | md5 | Use of weak MD5 hash |
| B608 | hardcoded_sql_expressions | SQL built with string formatting |

## Gate 2: Dependency Vulnerability Scanning

Your code might be secure, but **what about your dependencies?** If you're using a library with a known vulnerability, attackers can exploit it.

The famous **Log4Shell** vulnerability (CVE-2021-44228) affected millions of applications through a single Java logging library.

### How It Works

1. Tools read your dependency file (`pyproject.toml`, `package.json`, etc.)
2. They check each package version against databases of known vulnerabilities (CVEs)
3. If a vulnerable version is found, the build fails

### Popular Dependency Scanners

| Tool | Language | Free? |
|------|----------|-------|
| **pip-audit** | Python | ✅ |
| **safety** | Python | ✅ Community |
| **npm audit** | JavaScript | ✅ Built-in |
| **Dependabot** | Many | ✅ GitHub built-in |
| **Snyk** | Many | ✅ Free tier |

In [ ]:
# === Gate 2: Dependency vulnerability scanning ===
# Let's scan our pyproject.toml for known vulnerabilities

# First, let's look at what we're scanning
print("📦 Our dependencies (pyproject.toml):")
print("=" * 50)
with open("../pyproject.toml") as f:
    deps = f.read()
    print(deps)

print("\n🔍 Gate 2: Scanning dependencies for known vulnerabilities...")
print("=" * 70)

In [ ]:
# Run safety check (or pip-audit as alternative)
import subprocess
import sys

print("Running 'safety check' on installed packages...\n")

result = subprocess.run(
    [sys.executable, "-m", "safety", "check", "--output", "text"],
    capture_output=True,
    text=True,
)

output = result.stdout + result.stderr
print(output[:2000])  # show first 2000 chars

if result.returncode == 0:
    print("\n✅ GATE PASSED: No known vulnerabilities in dependencies.")
else:
    print("\n⚠️  Vulnerabilities found! In a real pipeline, this would block the merge.")
    print("   Fix: Update affected packages to patched versions.")

In [ ]:
# === DEMO: What a vulnerable dependency looks like ===
# Let's simulate finding a vulnerability in a dependency

simulated_vulns = [
    {
        "package": "flask",
        "installed": "2.3.0",
        "affected": "<2.3.2",
        "fixed_in": "2.3.2",
        "cve": "CVE-2023-30861",
        "severity": "HIGH",
        "description": "Cookie handling could allow session data leakage",
    },
    {
        "package": "requests",
        "installed": "2.28.0",
        "affected": "<2.31.0",
        "fixed_in": "2.31.0",
        "cve": "CVE-2023-32681",
        "severity": "MEDIUM",
        "description": "Proxy-Authorization header leaked to third-party hosts",
    },
]

print("📊 Example: Dependency Vulnerability Report")
print("=" * 70)
print("(Simulated — real results depend on your installed versions)\n")

for vuln in simulated_vulns:
    icon = "🔴" if vuln["severity"] == "HIGH" else "🟡"
    print(f"  {icon} {vuln['package']} {vuln['installed']}")
    print(f"     CVE: {vuln['cve']} ({vuln['severity']})")
    print(f"     Issue: {vuln['description']}")
    print(f"     Affected: {vuln['affected']}")
    print(f"     Fix: pip install {vuln['package']}>={vuln['fixed_in']}")
    print()

print("💡 Dependabot (built into GitHub) can automatically create PRs")
print("   to update vulnerable packages.")

## Gate 3: DAST — Dynamic Application Security Testing

**DAST** tests your **running application** by sending malicious requests and checking the responses.

| Feature | SAST | DAST |
|---------|------|------|
| **Scans** | Source code | Running application |
| **Requires** | Code access | Network access |
| **Finds** | Code-level bugs | Runtime vulnerabilities |
| **Speed** | Fast | Slower |
| **False positives** | Higher | Lower |
| **Example tools** | Bandit, Semgrep | OWASP ZAP, Burp Suite |

DAST is like hiring a robot pentester that automatically tries common attacks against your app.

In [ ]:
# === Gate 3: Simple DAST — automated vulnerability testing ===
# In production you'd use OWASP ZAP or Burp Suite.
# Here we build a simple scanner to demonstrate the concept.

import requests

BASE_URL = "http://localhost:5001"

class SimpleDAST:
    """A minimal DAST scanner for educational purposes.
    Real tools like OWASP ZAP have thousands of checks.
    """
    
    def __init__(self, base_url):
        self.base_url = base_url
        self.findings = []
    
    def test_sql_injection(self):
        """Test for SQL injection vulnerabilities."""
        payloads = [
            "' OR '1'='1' --",
            "' UNION SELECT 1,2,3 --",
            "'; DROP TABLE products; --",
        ]
        
        for payload in payloads:
            # Test vulnerable endpoint
            resp = requests.get(f"{self.base_url}/api/products/search", params={"q": payload})
            if resp.status_code == 200 and len(resp.json()) > 5:
                self.findings.append({
                    "type": "SQL Injection",
                    "severity": "CRITICAL",
                    "endpoint": "/api/products/search",
                    "payload": payload,
                    "evidence": f"Returned {len(resp.json())} results (expected few or error)",
                })
                break  # one finding is enough
            
            # Test safe endpoint (should not be vulnerable)
            resp_safe = requests.get(f"{self.base_url}/api/products/search/safe", params={"q": payload})
            if resp_safe.status_code == 200 and len(resp_safe.json()) > 5:
                self.findings.append({
                    "type": "SQL Injection",
                    "severity": "CRITICAL",
                    "endpoint": "/api/products/search/safe",
                    "payload": payload,
                    "evidence": "Safe endpoint also vulnerable!",
                })
    
    def test_xss(self):
        """Test for Cross-Site Scripting vulnerabilities."""
        xss_payload = '<script>alert(1)</script>'
        
        # Inject XSS via comment
        requests.post(f"{self.base_url}/api/comments", json={
            "user_id": 2, "product_id": 1, "content": xss_payload,
        })
        
        # Check if reflected in vulnerable endpoint
        resp = requests.get(f"{self.base_url}/comments/1")
        if "<script>" in resp.text:
            self.findings.append({
                "type": "Stored XSS",
                "severity": "HIGH",
                "endpoint": "/comments/1",
                "payload": xss_payload,
                "evidence": "Unescaped <script> tag found in response",
            })
        
        # Check safe endpoint
        resp_safe = requests.get(f"{self.base_url}/comments/1/safe")
        if "<script>" in resp_safe.text:
            self.findings.append({
                "type": "Stored XSS",
                "severity": "HIGH",
                "endpoint": "/comments/1/safe",
                "payload": xss_payload,
                "evidence": "Safe endpoint also has unescaped scripts!",
            })
    
    def test_ssrf(self):
        """Test for Server-Side Request Forgery."""
        internal_urls = [
            "http://localhost:5001/health",
            "http://127.0.0.1:6379",
        ]
        
        for url in internal_urls:
            resp = requests.get(f"{self.base_url}/api/fetch-url", params={"url": url})
            data = resp.json()
            if resp.status_code == 200 and "error" not in data:
                self.findings.append({
                    "type": "SSRF",
                    "severity": "CRITICAL",
                    "endpoint": "/api/fetch-url",
                    "payload": url,
                    "evidence": f"Server fetched internal URL (status: {data.get('status_code')})",
                })
                break
    
    def test_csrf(self):
        """Test for missing CSRF protection."""
        resp = requests.post(
            f"{self.base_url}/api/transfer",
            json={"from_user": "alice", "to_user": "attacker", "amount": 100},
            headers={"Origin": "http://evil.com"},
        )
        if resp.status_code == 200:
            self.findings.append({
                "type": "Missing CSRF Protection",
                "severity": "HIGH",
                "endpoint": "/api/transfer",
                "payload": "Cross-origin POST with no CSRF token",
                "evidence": "Transfer succeeded from evil.com origin",
            })
    
    def test_info_disclosure(self):
        """Test for information disclosure on login."""
        resp1 = requests.post(f"{self.base_url}/api/login", json={
            "username": "alice", "password": "wrong"
        })
        resp2 = requests.post(f"{self.base_url}/api/login", json={
            "username": "nonexistent_xyz", "password": "wrong"
        })
        
        if resp1.status_code != resp2.status_code:
            self.findings.append({
                "type": "Information Disclosure",
                "severity": "MEDIUM",
                "endpoint": "/api/login",
                "payload": "Different error for existing vs non-existing user",
                "evidence": f"Existing user: {resp1.status_code}, Non-existing: {resp2.status_code}",
            })
    
    def run_all(self):
        """Run all DAST checks."""
        self.findings = []
        
        checks = [
            ("SQL Injection", self.test_sql_injection),
            ("XSS", self.test_xss),
            ("SSRF", self.test_ssrf),
            ("CSRF", self.test_csrf),
            ("Info Disclosure", self.test_info_disclosure),
        ]
        
        for name, check_fn in checks:
            print(f"  Checking {name}...", end=" ")
            try:
                check_fn()
                print("done")
            except Exception as e:
                print(f"error: {e}")
        
        return self.findings


# Run the DAST scanner
print("🔍 Gate 3: DAST — Scanning the running application")
print("=" * 70)

scanner = SimpleDAST(BASE_URL)
findings = scanner.run_all()

print(f"\n📊 Results: {len(findings)} vulnerabilities found\n")
for f in findings:
    icon = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡"}.get(f["severity"], "⚪")
    print(f"  {icon} [{f['severity']}] {f['type']}")
    print(f"     Endpoint: {f['endpoint']}")
    print(f"     Evidence: {f['evidence']}")
    print()

critical = sum(1 for f in findings if f["severity"] == "CRITICAL")
high = sum(1 for f in findings if f["severity"] == "HIGH")

if critical > 0 or high > 0:
    print(f"❌ GATE FAILED: {critical} Critical, {high} High findings. Merge blocked.")
else:
    print("✅ GATE PASSED: No critical/high vulnerabilities found.")

## Gate 4: Security Sign-Off Process

Automated tools catch most issues, but some require **human judgment**. At Microsoft, the SDL includes a formal security review before major releases.

### When Manual Review is Required

| Change Type | Automated Only? | Manual Review? |
|-------------|----------------|----------------|
| Bug fix in existing feature | ✅ Automated is enough | ❌ |
| New API endpoint | ✅ Automated | ⚠️ Recommended |
| New authentication flow | ✅ Automated | ✅ Required |
| Handling financial data | ✅ Automated | ✅ Required |
| Changes to encryption | ✅ Automated | ✅ Required |
| Third-party integration | ✅ Automated | ✅ Required |

### The Microsoft SDL Security Review Checklist

```
□ Threat model reviewed and updated
□ All STRIDE threats have mitigations
□ SAST scan passes with no high-severity issues
□ Dependency scan passes (no known CVEs)
□ DAST scan passes (no critical findings)
□ Penetration test completed (for major releases)
□ Secrets are not hardcoded (vault or managed identity)
□ Logging and monitoring in place
□ Incident response plan documented
□ Security team sign-off obtained
```

In [ ]:
# === DEMO: Security Gate Pipeline Summary ===
# Let's combine all gates into a single pipeline report

print("📋 Security Gate Pipeline Report")
print("=" * 70)
print(f"Application: Security Demo Flask App")
print(f"Scan Date: 2024-01-15")
print(f"Branch: feature/new-api-endpoint")
print()

gates = [
    {
        "name": "Gate 1: SAST (Bandit)",
        "status": "FAIL",
        "details": "3 High, 2 Medium findings",
        "action": "Fix SQL injection on line 45, remove hardcoded secrets",
    },
    {
        "name": "Gate 2: Dependency Scan (safety)",
        "status": "PASS",
        "details": "0 vulnerabilities in 11 packages",
        "action": "None required",
    },
    {
        "name": "Gate 3: DAST (automated scan)",
        "status": "FAIL",
        "details": f"{len(findings)} vulnerabilities found",
        "action": "Fix SQL injection, XSS, SSRF, CSRF endpoints",
    },
    {
        "name": "Gate 4: Security Review",
        "status": "PENDING",
        "details": "Awaiting security team review",
        "action": "Schedule review after automated gates pass",
    },
]

all_pass = True
for gate in gates:
    icon = {"PASS": "✅", "FAIL": "❌", "PENDING": "⏳"}[gate["status"]]
    print(f"{icon} {gate['name']}: {gate['status']}")
    print(f"   {gate['details']}")
    print(f"   Action: {gate['action']}")
    print()
    if gate["status"] == "FAIL":
        all_pass = False

print("=" * 70)
if all_pass:
    print("🎉 ALL GATES PASSED — Ready for merge!")
else:
    print("🚫 MERGE BLOCKED — Fix failing gates before merging.")
    print("\nThis is exactly how enterprise CI/CD pipelines work:")
    print("no code reaches production until ALL security gates pass.")

## Example: GitHub Actions Security Pipeline

Here's what a real security gate pipeline looks like in GitHub Actions:

```yaml
# .github/workflows/security.yml
name: Security Gates

on:
  pull_request:
    branches: [main]

jobs:
  sast:
    name: "Gate 1: SAST (Bandit)"
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install bandit
      - run: bandit -r app/ -f json -o bandit-report.json
      - uses: actions/upload-artifact@v4
        with:
          name: bandit-report
          path: bandit-report.json

  dependency-scan:
    name: "Gate 2: Dependency Scan"
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install safety
      - run: uv sync
      - run: safety check

  dast:
    name: "Gate 3: DAST (OWASP ZAP)"
    runs-on: ubuntu-latest
    needs: [sast, dependency-scan]  # only run if earlier gates pass
    steps:
      - uses: actions/checkout@v4
      - run: docker-compose up -d
      - run: sleep 10  # wait for app to start
      - uses: zaproxy/action-baseline@v0.10.0
        with:
          target: "http://localhost:5001"

  security-review:
    name: "Gate 4: Security Sign-Off"
    runs-on: ubuntu-latest
    needs: [sast, dependency-scan, dast]
    environment: security-review  # requires manual approval
    steps:
      - run: echo "Security team has approved this release"
```

The `environment: security-review` line means a human from the security team must click "Approve" in GitHub before the pipeline continues.

## Penetration Testing

**Penetration testing** (pen testing) is when security experts manually try to break into your system. It goes beyond automated tools because humans can:
- Chain multiple low-risk issues into a critical exploit
- Think creatively about business logic flaws
- Find issues that automated tools miss

### Types of Pen Tests

| Type | Tester Knowledge | When to Use |
|------|-----------------|-------------|
| **Black box** | No knowledge of the system | Simulates an external attacker |
| **White box** | Full access to source code | Most thorough, finds the most issues |
| **Gray box** | Partial knowledge (e.g., API docs) | Balance of realism and thoroughness |

### Microsoft SDL Pen Testing Requirements

- Pen testing is **required** before major releases
- Must be done by an **independent team** (not the developers)
- Findings must be **tracked and remediated** before release
- Re-test after fixes to confirm they work

## 🔑 Key Takeaways

1. **Automate everything you can** — SAST, dependency scanning, and basic DAST should run on every PR
2. **SAST finds code-level bugs** early (Bandit for Python) — fast and catches SQL injection, hardcoded secrets
3. **Dependency scanning** catches vulnerabilities in your libraries — run `safety check` or `pip-audit`
4. **DAST tests your running app** — sends real attacks to find runtime vulnerabilities
5. **Security gates block merges** — no code reaches production without passing all gates
6. **Human review** is still needed — for authentication changes, financial operations, and cryptography
7. **Penetration testing** validates everything before major releases

### Enterprise Security Pipeline Summary

| Gate | Tool | Runs When | Blocks Merge? |
|------|------|-----------|---------------|
| SAST | Bandit, Semgrep, CodeQL | Every PR | ✅ Yes (High+ findings) |
| Dependencies | safety, Dependabot | Every PR | ✅ Yes (known CVEs) |
| DAST | OWASP ZAP, Burp Suite | After deploy to staging | ✅ Yes (Critical findings) |
| Pen Test | Manual by security team | Before major releases | ✅ Yes (Critical findings) |
| Sign-Off | Security team review | Before production deploy | ✅ Yes (must be approved) |

## 🏁 Lab Complete!

You've learned the complete enterprise security review process:
1. **Threat Modeling** — identify what can go wrong (STRIDE)
2. **Vulnerability Knowledge** — understand and fix common attacks
3. **Secrets Management** — protect credentials properly
4. **Security Gates** — automate checks in CI/CD

These practices are used daily at Microsoft, Google, Amazon, and every major tech company.